In [ ]:
import pandas as pd
import difflib
import re

df = pd.read_csv('../data/addresses.csv')
print(f"loaded {len(df)} rows")
df.head()

In [ ]:
def normalize(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[^\w\s]', ' ', text)
    return " ".join(text.strip().lower().split())

df['addr_norm'] = df['address'].apply(normalize)
df['match_norm'] = df['matched_address'].apply(normalize)

In [ ]:
def seq_score(row):
    return difflib.SequenceMatcher(None, row['addr_norm'], row['match_norm']).ratio()

df['score_seq'] = df.apply(seq_score, axis=1)
df[['address', 'matched_address', 'score_seq']].head()

In [ ]:
# Token Set Ratio (Jaccard-like)
def token_score(row):
    a = set(row['addr_norm'].split())
    b = set(row['match_norm'].split())
    if not a or not b: return 0.0
    return 2.0 * len(a.intersection(b)) / (len(a) + len(b))

df['score_token'] = df.apply(token_score, axis=1)

In [ ]:
# Substr Match
def sub_score(row):
    a, b = row['addr_norm'], row['match_norm']
    if not a or not b: return 0.0
    
    if a in b or b in a:
        ratio = min(len(a), len(b)) / max(len(a), len(b))
        return 0.8 * ratio
    return 0.0

df['score_sub'] = df.apply(sub_score, axis=1)

In [ ]:
df['score_hybrid'] = df[['score_seq', 'score_token', 'score_sub']].max(axis=1)
df['error'] = abs(df['score_hybrid'] - df['semantic_similarity'])
print(f"Avg err: {df['error'].mean():.4f}")

In [ ]:
df.sort_values('error', ascending=False)[['address', 'matched_address', 'semantic_similarity', 'score_hybrid']].head(15)